# LAB 08 - TravelOps
## Notebook: 00_seed_raw_data

Purpose:
Seeds selected `samples.wanderbricks` source tables into the Terraform-owned raw landing Volume.

Business purpose:
Creates repeatable raw travel booking data so CI/CD can prove the same application promotes from DEV to PROD.

Technical purpose:
Verifies the DAB target schema and Terraform-owned raw Volume, then overwrites deterministic Parquet folders used by Auto Loader.

Inputs:
- samples.wanderbricks.bookings
- samples.wanderbricks.booking_updates
- samples.wanderbricks.payments
- samples.wanderbricks.users
- samples.wanderbricks.properties
- samples.wanderbricks.reviews
- samples.wanderbricks.destinations

Outputs:
- /Volumes/<raw_volume_catalog>/<raw_volume_schema>/<raw_volume>/raw/<source_table>/

Tables/files affected:
Only raw Parquet folders are overwritten. Schema and Volume metadata are owned by Terraform.

Environment variables/widgets used:
`target_catalog`, `target_schema`, `raw_volume_catalog`, `raw_volume_schema`, `raw_volume_name`, `raw_volume_type`, `raw_volume_storage_location`, `source_catalog`, `source_schema`, `seed_limit`.

Creates/modifies data:
Yes. It writes deterministic raw landing files and is safe to rerun.

Dependencies/prerequisites:
The DAB target schema must exist, Terraform must have already created the raw Volume, and the deployer must have read access to `samples.wanderbricks`.

Expected result:
Each configured source table has one raw Parquet file, at a name derived from `seed_limit`, in the target Volume. `bookings`, `users`, `properties` and `destinations` are sampled independently by row order. `booking_updates`, `payments` and `reviews` are booking-scoped event tables, so they are instead filtered to the exact `booking_id` set sampled from `bookings`, keeping every related event for a sampled booking instead of an independent, referentially-inconsistent row-count cap.

Idempotency note:
This overwrite is deterministic in *content* (the same source query returns the same rows on every run). Ingestion idempotency across reruns is now handled at the source, not just downstream: `_write_stable_seed`/`_seed_file_name` write each table to one Parquet file at a path derived from `seed_limit`, instead of letting Spark allocate a fresh, uniquely-named file on every write. Auto Loader's default `cloudFiles.allowOverwrites=false` means it will not reprocess a path it has already ingested, even if that path is later overwritten with new bytes — so a rerun with an unchanged `seed_limit` reuses the same path and Auto Loader skips it, instead of re-ingesting it as new. Changing `seed_limit` changes the path, so a deliberate configuration change is still picked up. This is based on Auto Loader's documented `cloudFiles.allowOverwrites` behavior; it has not been empirically validated by an actual run in this repository, and a controlled DEV run is the recommended way to confirm it before relying on it for Azure PROD (see `evidence/lab08_production_remediation_plan.md`). It also does not cover every possible content change (for example, `samples.wanderbricks` itself changing upstream without any local `seed_limit`/config change keeps the same path and would not be picked up), and it does not retroactively deduplicate Bronze rows already accumulated from runs before this fix shipped. `pipeline/silver.py` still deduplicates `payments_silver` on a composite business key and keeps only the latest row per `booking_id` in `current_bookings_silver` as a defense-in-depth backstop for those residual cases — Silver deduplication is a backstop, not the primary mechanism that stops Bronze from accumulating duplicates.

Failure behavior:
The notebook fails fast when required parameters are missing, when the Terraform-owned raw Volume is absent, or when source/target privileges are insufficient.

Environment classification:
Safe for personal_dev and personal_prod runs. Azure PROD job execution is intentionally deferred until explicitly authorized.


### Step 1 - Resolve target configuration

This cell reads Databricks widgets supplied by the Bundle job. It validates required values before any write occurs so configuration errors fail before partial data is created.

In [ ]:
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("raw_volume_catalog", "")
dbutils.widgets.text("raw_volume_schema", "")
dbutils.widgets.text("raw_volume_name", "lab08_dev_travelops_raw")
dbutils.widgets.text("raw_volume_type", "MANAGED")
dbutils.widgets.text("raw_volume_storage_location", "")
dbutils.widgets.text("source_catalog", "samples")
dbutils.widgets.text("source_schema", "wanderbricks")
dbutils.widgets.text("seed_limit", "25000")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
raw_volume_catalog = dbutils.widgets.get("raw_volume_catalog")
raw_volume_schema = dbutils.widgets.get("raw_volume_schema")
raw_volume_name = dbutils.widgets.get("raw_volume_name")
raw_volume_type = dbutils.widgets.get("raw_volume_type").upper()
raw_volume_storage_location = dbutils.widgets.get("raw_volume_storage_location")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
seed_limit = int(dbutils.widgets.get("seed_limit"))

required = {
    "target_catalog": target_catalog,
    "target_schema": target_schema,
    "raw_volume_catalog": raw_volume_catalog,
    "raw_volume_schema": raw_volume_schema,
    "raw_volume_name": raw_volume_name,
    "source_catalog": source_catalog,
    "source_schema": source_schema,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError(f"Missing required widgets: {missing}")
if raw_volume_type == "EXTERNAL" and not raw_volume_storage_location:
    raise ValueError("External raw volume requires raw_volume_storage_location")


### Step 2 - Verify target schema and Terraform-owned raw Volume

This cell performs read-only metadata checks. It intentionally does not create or alter the raw Volume schema or raw Volume because Terraform owns that layer; the application target schema is checked separately.

In [ ]:
target_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{target_catalog}` LIKE '{target_schema}'").count() == 1
if not target_schema_exists:
    raise ValueError(f"Required DAB target schema does not exist: {target_catalog}.{target_schema}")

raw_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{raw_volume_catalog}` LIKE '{raw_volume_schema}'").count() == 1
if not raw_schema_exists:
    raise ValueError(f"Required Terraform-referenced raw schema does not exist: {raw_volume_catalog}.{raw_volume_schema}")

volume_rows = spark.sql(f"SHOW VOLUMES IN `{raw_volume_catalog}`.`{raw_volume_schema}` LIKE '{raw_volume_name}'").collect()
if len(volume_rows) != 1:
    raise ValueError(f"Required Terraform-owned raw Volume is missing: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")

print(f"Verified DAB target schema: {target_catalog}.{target_schema}")
print(f"Verified Terraform-owned raw Volume: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")


### Step 3 - Seed deterministic raw Parquet folders

This cell reads the discovered Wanderbricks tables and overwrites deterministic target folders. Overwrite mode prevents uncontrolled duplicate files across reruns while keeping the raw contract file-based for Auto Loader.

In [ ]:
source_tables = [
    "bookings",
    "booking_updates",
    "payments",
    "users",
    "properties",
    "reviews",
    "destinations",
]

# booking_updates, payments and reviews reference booking_id. Sampling each
# of these independently by row order (as bookings/users/properties/
# destinations still are) breaks that relationship: a booking's related
# payment or update rows can fall outside a same-sized but differently
# ordered independent sample, which previously produced "missing payment"
# rows caused purely by sampling rather than by the source data. Sampling
# bookings first and then filtering these three tables to that exact
# booking_id set keeps every related event for a sampled booking.
BOOKING_SCOPED_TABLES = {"booking_updates", "payments", "reviews"}

bookings_source = f"`{source_catalog}`.`{source_schema}`.`bookings`"
bookings_df = spark.table(bookings_source)
bookings_sample_df = bookings_df.orderBy(*bookings_df.columns[:1]).limit(seed_limit)
sampled_booking_ids_df = bookings_sample_df.select("booking_id")


def _seed_file_name(current_seed_limit: int) -> str:
    """Deterministic Parquet file name for this run's raw seed.

    Spark's DataFrameWriter always allocates a fresh, uniquely-named part
    file on every write, even in overwrite mode, so writing straight to
    target_path (the previous implementation) produced a new file path on
    every run regardless of whether the content changed. Auto Loader's
    checkpoint tracks already-ingested input by file path, and its default
    cloudFiles.allowOverwrites=false means it will not reprocess a path it
    has already seen, even if that path is later overwritten with new
    bytes. Writing to a STABLE path derived from seed_limit is what makes a
    rerun with unchanged seed_limit get skipped by Auto Loader instead of
    re-ingested as new -- this is a genuine ingestion-idempotency fix, not
    a downstream mitigation. Changing seed_limit changes this file name, so
    a deliberate configuration change is still picked up as new content.

    Limitation: this does not detect every possible content change -- for
    example, if samples.wanderbricks itself changes upstream without any
    local seed_limit/config change, the path stays the same and Auto Loader
    will not pick up the new content. That residual case (and any Bronze
    duplication already accumulated before this fix shipped) is why
    pipeline/silver.py still deduplicates payments_silver on a composite
    business key and current_bookings_silver keeps only the latest row per
    booking_id -- those remain a defense-in-depth backstop, not the primary
    idempotency mechanism.
    """

    return f"seed-{current_seed_limit}.snappy.parquet"


def _write_stable_seed(df, target_path: str, staging_root: str, table_name: str) -> None:
    """Overwrite target_path with df as a single, stably-named Parquet file.

    Writes to a staging path outside the raw/<table>/ tree Auto Loader
    watches, confirms exactly one part file was produced (via coalesce(1)),
    clears target_path, then moves the part file into place under the
    deterministic name from _seed_file_name. This preserves the original
    "deterministic overwrite, no uncontrolled duplicate files" contract
    while also giving Auto Loader a stable path to recognize as
    already-processed on a rerun with unchanged seed_limit.
    """

    staging_path = f"{staging_root}/{table_name}"
    try:
        dbutils.fs.rm(staging_path, recurse=True)
    except Exception:
        pass

    df.coalesce(1).write.mode("overwrite").format("parquet").save(staging_path)
    part_files = [f.path for f in dbutils.fs.ls(staging_path) if f.name.startswith("part-")]
    if len(part_files) != 1:
        raise AssertionError(
            f"Expected exactly one staged part file for {table_name}, found {len(part_files)}: {part_files}"
        )

    try:
        dbutils.fs.rm(target_path, recurse=True)
    except Exception:
        pass

    final_path = f"{target_path}/{_seed_file_name(seed_limit)}"
    dbutils.fs.mv(part_files[0], final_path)
    dbutils.fs.rm(staging_path, recurse=True)


staging_root = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/_seed_staging"

seed_summary = []
for table_name in source_tables:
    source_table = f"`{source_catalog}`.`{source_schema}`.`{table_name}`"
    target_path = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/raw/{table_name}"
    if table_name == "bookings":
        df = bookings_sample_df
    elif table_name in BOOKING_SCOPED_TABLES:
        df = spark.table(source_table).join(sampled_booking_ids_df, "booking_id", "left_semi")
    else:
        source_df = spark.table(source_table)
        df = source_df.orderBy(*source_df.columns[:1]).limit(seed_limit)
    row_count = df.count()
    _write_stable_seed(df, target_path, staging_root, table_name)
    seed_summary.append((table_name, row_count, target_path))

display(spark.createDataFrame(seed_summary, "table_name STRING, row_count LONG, target_path STRING"))
